# AI Hub 데이터셋 기반 STT 성능 벤치마크 — faster-whisper(small) vs SenseVoiceSmall

데이터셋: **141. 의료진 및 환자 음성** (datasetkey: `71411`)
- 라벨링데이터: `48745`
- 원천데이터: `48746` ~ `48758`

AI Hub는 해외/클라우드 IP에서의 다운로드를 차단해 Colab에서 `aihubshell`로 직접 받을 수 없습니다. 대신 로컬에 다운로드한 뒤 `rclone`으로 Google Drive(`colab/audiodata`)에 업로드해둘았고, 이 노트북은 Drive를 마운트해 바로 사용합니다.

한국어 의료 음성 데이터로 두 STT 모델의 실시간성과 정확도를 비교합니다.

**측정 지표**
- Latency (초): 오디오 1개당 추론 소요 시간
- RTF (Real-Time Factor) = 추론 시간 / 오디오 길이 — 1보다 작을수록 실시간보다 빠름
- CER (Character Error Rate): `jiwer` 기반 한국어 음절 오류율

**비교 대상**
- `faster-whisper` (`small`, GPU, float16)
- `SenseVoiceSmall` (FunASR/ModelScope, GPU)

실행 전에 Colab 상단 메뉴에서 **런타임 유형을 GPU로 변경**하세요 (런타임 > 런타임 유형 변경).

## 1. 환경 설정 및 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q faster-whisper funasr modelscope torchaudio jiwer librosa soundfile pandas tabulate tqdm

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU가 잡히지 않았습니다 — 상단 메뉴 [런타임 > 런타임 유형 변경]에서 GPU를 선택하세요.")

### 압축 해제

`colab/audiodata` 밑에 올라온 Validation 원천/라벨 zip을 I/O 속도 확보를 위해 Drive가 아닌 로컬(`/content/data`) 디스크에 푸는다.
경로가 다르면 `DRIVE_DATA_ROOT`만 수정하면 된다.

In [ ]:
import os

DRIVE_DATA_ROOT = "/content/drive/MyDrive/colab/audiodata/비대면 진료를 위한 의료진 및 환자 음성/Validation"
SOURCE_ZIP = f"{DRIVE_DATA_ROOT}/[V원천]의료진_간호사_1.zip"
LABEL_ZIP = f"{DRIVE_DATA_ROOT}/[V]라벨링데이터.zip"

EXTRACT_ROOT = "/content/data"
AUDIO_DIR = f"{EXTRACT_ROOT}/source"
LABEL_DIR = f"{EXTRACT_ROOT}/label"

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)

!unzip -q -o -O UTF-8 "{SOURCE_ZIP}" -d "{AUDIO_DIR}"
!unzip -q -o -O UTF-8 "{LABEL_ZIP}" -d "{LABEL_DIR}"

print("Source files:", sum(len(f) for _, _, f in os.walk(AUDIO_DIR)))
print("Label files:", sum(len(f) for _, _, f in os.walk(LABEL_DIR)))

## 2. 모델 준비

In [ ]:
from faster_whisper import WhisperModel

fw_model = WhisperModel("small", device="cuda", compute_type="float16")

try:
    from faster_whisper import BatchedInferencePipeline
    fw_pipeline = BatchedInferencePipeline(model=fw_model)
    FW_BATCHED = True
    print("faster-whisper(small) loaded with BatchedInferencePipeline (파일 내부 VAD 청크 배치 디코딩).")
except ImportError:
    fw_pipeline = None
    FW_BATCHED = False
    print("faster-whisper(small) loaded (이 버전은 BatchedInferencePipeline 미지원 — 일반 모드로 동작).")

In [ ]:
from funasr import AutoModel
from funasr.utils.postprocess_utils import rich_transcription_postprocess

sv_model = AutoModel(
    model="iic/SenseVoiceSmall",
    trust_remote_code=True,
    device="cuda:0",
)
print("SenseVoiceSmall loaded.")

## 3. 오디오-라벨 페어링

In [ ]:
from pathlib import Path

AUDIO_EXTS = (".wav", ".flac", ".pcm")
LABEL_EXTS = (".json", ".txt")

audio_files = [p for p in Path(AUDIO_DIR).rglob("*") if p.suffix.lower() in AUDIO_EXTS]
label_files = [p for p in Path(LABEL_DIR).rglob("*") if p.suffix.lower() in LABEL_EXTS]

label_by_stem = {p.stem: p for p in label_files}

pairs = []
unmatched = 0
for audio_path in audio_files:
    label_path = label_by_stem.get(audio_path.stem)
    if label_path is None:
        unmatched += 1
        continue
    pairs.append((audio_path, label_path))

print(f"오디오 파일: {len(audio_files)}개")
print(f"라벨 파일: {len(label_files)}개")
print(f"매칭된 쌍: {len(pairs)}개  (매칭 안 된 오디오: {unmatched}개)")

### 라벨 스키마 확인

아래 셀에서 라벨 파일 하나를 열어 실제 필드 구조를 확인하세요. 바로 아래 `load_transcript()`가 흔한 AI Hub 필드명을 자동으로 찾지만, 데이터셋마다 스키마가 달라 실패할 수 있습니다 — 그럴 경우 출력된 raw 내용을 보고 `_TEXT_KEY_CANDIDATES`에 실제 필드명을 추가하세요.

In [ ]:
import json

if pairs:
    sample_label_path = pairs[0][1]
    print("Sample label file:", sample_label_path)
    if sample_label_path.suffix.lower() == ".json":
        with open(sample_label_path, encoding="utf-8") as f:
            print(json.dumps(json.load(f), ensure_ascii=False, indent=2)[:2000])
    else:
        print(sample_label_path.read_text(encoding="utf-8")[:2000])
else:
    print("매칭된 쌍이 없습니다 — 압축 해제 경로/파일명을 확인하세요.")

In [ ]:
import re

# AI Hub STT 라벨 JSON은 데이터셋마다 필드명이 다릅니다.
# 위 셀에서 출력된 실제 구조를 보고, 여기에 맞는 키를 추가하세요.
_TEXT_KEY_CANDIDATES = ["전사정보", "transcription", "text", "TransLabelText", "orgtext", "standard"]

def _find_text_value(obj, depth=0):
    if depth > 6:
        return None
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        for key in _TEXT_KEY_CANDIDATES:
            if key in obj:
                found = _find_text_value(obj[key], depth + 1)
                if found:
                    return found
        for value in obj.values():
            found = _find_text_value(value, depth + 1)
            if found:
                return found
    if isinstance(obj, list):
        for item in obj:
            found = _find_text_value(item, depth + 1)
            if found:
                return found
    return None

def load_transcript(label_path: Path) -> str:
    if label_path.suffix.lower() == ".txt":
        return label_path.read_text(encoding="utf-8").strip()
    with open(label_path, encoding="utf-8") as f:
        data = json.load(f)
    text = _find_text_value(data)
    if not text:
        raise ValueError(f"'{label_path}'에서 전사 텍스트를 찾지 못했습니다 — _TEXT_KEY_CANDIDATES에 실제 필드명을 추가하세요.")
    return text.strip()

def normalize_text(text: str) -> str:
    # CER 비교 전 공백/문장부호 정규화 (필요에 따라 조정)
    text = re.sub(r"[^\uac00-\ud7a30-9a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## 4. 벤치마크 실행

### 왜 스레딩이 아니라 배치인가

Colab은 GPU가 1개뿐이고 CUDA 커널 실행은 기본적으로 하나의 스트림에서 직렬화되기 때문에, Python 스레드(`ThreadPoolExecutor` 등)로 파일 여러 개를 "동시에" GPU에 넣어도 실제 GPU 연산 자체는 병렬로 겹치지 않는다. 진짜 속도 향상은 **한 번의 forward pass에 여러 데이터를 묶어서 넣는 배치 처리**에서 나온다:

- **faster-whisper**: `BatchedInferencePipeline`으로 파일 내부를 VAD로 잘라 청크 단위 배치 디코딩. 파일이 길수록(수 분 이상) 효과가 크고, 원래 짧은 파일이면 청크가 거의 안 나와 이득이 적을 수 있다.
- **SenseVoice(FunASR)**: 파일을 하나씩 `generate()`에 넣던 것을, 여러 파일을 리스트로 묶어 한 번에 배치 추론하도록 변경. 파일 단위 순차 호출보다 GPU 활용률이 크게 오른다.

두 방식 모두 GPU 메모리를 더 쓰므로, OOM이 나면 아래 `FW_BATCH_SIZE` / `SV_BATCH_SIZE`를 줄이세요. (추가로 시도해볼 수 있는 것: faster-whisper `beam_size`를 낮추거나 `compute_type="int8_float16"`으로 바꾸면 더 빨라지지만 CER이 약간 나빠질 수 있음 — 정확도 지표를 보는 벤치마크라 기본값은 그대로 뒀다.)

In [ ]:
import random
import soundfile as sf

random.seed(42)
SAMPLE_SIZE = 50  # None으로 바꾸면 매칭된 쌍 전체 실행 (오래 걸릴 수 있음)
FW_BATCH_SIZE = 16  # faster-whisper: 파일 내부 VAD 청크 배치 크기
SV_BATCH_SIZE = 16  # SenseVoice: 한 번에 배치로 넣을 파일 개수 (크게 잡을수록 빠르지만 GPU 메모리 더 사용, OOM 나면 줄이기)

sampled_pairs = pairs if SAMPLE_SIZE is None else random.sample(pairs, min(SAMPLE_SIZE, len(pairs)))

# 참조 텍스트/정규화 결과/오디오 길이는 두 모델이 동일한 파일을 평가하므로 한 번만 계산해 재사용한다.
# (모델별로 각각 다시 읽고 파싱하면 라벨 I/O와 정규화가 그대로 중복된다.)
eval_items = []
for audio_path, label_path in sampled_pairs:
    try:
        reference = load_transcript(label_path)
        eval_items.append({
            "audio_path": audio_path,
            "reference": reference,
            "ref_norm": normalize_text(reference),
            "duration_sec": sf.info(str(audio_path)).duration,
        })
    except Exception as e:
        print(f"전처리 실패, 이 파일은 벤치마크에서 제외: {audio_path.name} ({e})")

print(f"벤치마크 대상: {len(eval_items)}개")

In [ ]:
def infer_faster_whisper(audio_path: str) -> str:
    if FW_BATCHED:
        segments, _ = fw_pipeline.transcribe(audio_path, language="ko", beam_size=5, batch_size=FW_BATCH_SIZE)
    else:
        segments, _ = fw_model.transcribe(audio_path, language="ko", beam_size=5)
    return "".join(seg.text for seg in segments)

def infer_sensevoice(audio_path: str) -> str:
    """단일 파일 추론 — 배치 처리 실패 시 폴백용으로 사용."""
    result = sv_model.generate(
        input=audio_path,
        cache={},
        language="auto",
        use_itn=True,
        batch_size_s=60,
    )
    return rich_transcription_postprocess(result[0]["text"])

def infer_sensevoice_batch(audio_paths: list) -> list:
    """여러 파일을 한 번에 배치로 추론 — 파일을 하나씩 넣는 것보다 GPU 활용률이 훨씬 높다."""
    results = sv_model.generate(
        input=audio_paths,
        cache={},
        language="auto",
        use_itn=True,
        batch_size_s=300,
        batch_size_threshold_s=60,
    )
    return [rich_transcription_postprocess(r["text"]) for r in results]

In [ ]:
import time
from jiwer import cer as jiwer_cer
from tqdm.auto import tqdm

def run_benchmark(model_name: str, infer_fn, items):
    """파일 단위 순차 처리 (faster-whisper: BatchedInferencePipeline이 파일 내부를 청크 배치 디코딩)."""
    rows = []
    rtf_sum = 0.0
    cer_sum = 0.0
    cer_count = 0
    progress = tqdm(items, desc=model_name, unit="file")
    for item in progress:
        audio_path = item["audio_path"]
        try:
            start = time.perf_counter()
            hypothesis = infer_fn(str(audio_path))
            latency_sec = time.perf_counter() - start

            duration_sec = item["duration_sec"]
            hyp_norm = normalize_text(hypothesis)
            error_rate = jiwer_cer(item["ref_norm"], hyp_norm) if item["ref_norm"] else float("nan")
            rtf = latency_sec / duration_sec if duration_sec > 0 else float("nan")

            rows.append({
                "model": model_name,
                "file": audio_path.name,
                "duration_sec": duration_sec,
                "latency_sec": latency_sec,
                "rtf": rtf,
                "cer": error_rate,
                "reference": item["reference"],
                "hypothesis": hypothesis,
            })

            rtf_sum += rtf
            if error_rate == error_rate:  # NaN이 아닐 때만 누적
                cer_sum += error_rate
                cer_count += 1
            progress.set_postfix(
                avg_rtf=f"{rtf_sum / len(rows):.2f}",
                avg_cer=f"{(cer_sum / cer_count):.2f}" if cer_count else "n/a",
            )
        except Exception as e:
            progress.write(f"[{model_name}] {audio_path.name} 처리 실패: {e}")
    return rows


def run_benchmark_sensevoice_batched(items, batch_size: int = 16):
    """여러 파일을 한 번에 SenseVoice에 넣어 배치로 디코딩 (파일 단위 순차 호출보다 GPU 활용률이 훨씬 높음).
    파일별 latency는 배치 전체 wall time을 오디오 길이 비율로 배분한 값이다."""
    rows = []
    rtf_sum = 0.0
    cer_sum = 0.0
    cer_count = 0
    chunks = [items[i:i + batch_size] for i in range(0, len(items), batch_size)]
    progress = tqdm(chunks, desc="SenseVoiceSmall (batched)", unit="batch")

    def _record(item, duration_sec, latency_sec, hypothesis):
        nonlocal rtf_sum, cer_sum, cer_count
        hyp_norm = normalize_text(hypothesis)
        error_rate = jiwer_cer(item["ref_norm"], hyp_norm) if item["ref_norm"] else float("nan")
        rtf = latency_sec / duration_sec if duration_sec > 0 else float("nan")
        rows.append({
            "model": "SenseVoiceSmall",
            "file": item["audio_path"].name,
            "duration_sec": duration_sec,
            "latency_sec": latency_sec,
            "rtf": rtf,
            "cer": error_rate,
            "reference": item["reference"],
            "hypothesis": hypothesis,
        })
        rtf_sum += rtf
        if error_rate == error_rate:
            cer_sum += error_rate
            cer_count += 1

    for chunk in progress:
        try:
            audio_paths = [str(it["audio_path"]) for it in chunk]

            start = time.perf_counter()
            hypotheses = infer_sensevoice_batch(audio_paths)
            batch_latency_sec = time.perf_counter() - start

            total_duration = sum(it["duration_sec"] for it in chunk) or 1.0
            for item, hypothesis in zip(chunk, hypotheses):
                latency_share_sec = batch_latency_sec * (item["duration_sec"] / total_duration)
                _record(item, item["duration_sec"], latency_share_sec, hypothesis)
        except Exception as e:
            progress.write(f"배치 처리 실패, 이 배치는 파일 단위로 재시도: {e}")
            for item in chunk:
                try:
                    start = time.perf_counter()
                    hypothesis = infer_sensevoice(str(item["audio_path"]))
                    latency_sec = time.perf_counter() - start
                    _record(item, item["duration_sec"], latency_sec, hypothesis)
                except Exception as e2:
                    progress.write(f"[SenseVoiceSmall] {item['audio_path'].name} 처리 실패: {e2}")

        if rows:
            progress.set_postfix(
                files=len(rows),
                avg_rtf=f"{rtf_sum / len(rows):.2f}",
                avg_cer=f"{(cer_sum / cer_count):.2f}" if cer_count else "n/a",
            )
    return rows


fw_rows = run_benchmark("faster-whisper-small", infer_faster_whisper, eval_items)
sv_rows = run_benchmark_sensevoice_batched(eval_items, batch_size=SV_BATCH_SIZE)

## 5. 결과 집계 및 저장

In [ ]:
import pandas as pd
from tabulate import tabulate

results_df = pd.DataFrame(fw_rows + sv_rows)

summary_df = (
    results_df
    .groupby("model")[["duration_sec", "latency_sec", "rtf", "cer"]]
    .mean()
    .rename(columns={
        "duration_sec": "avg_duration_sec",
        "latency_sec": "avg_latency_sec",
        "rtf": "avg_rtf",
        "cer": "avg_cer",
    })
)

print(tabulate(summary_df, headers="keys", tablefmt="github", floatfmt=".4f"))

results_df.to_csv("stt_benchmark_results.csv", index=False, encoding="utf-8-sig")
print("\nSaved: stt_benchmark_results.csv")

### 결과 표 해석

| 컬럼 | 의미 |
| --- | --- |
| `model` | 비교 대상 STT 모델명 (`faster-whisper-small` / `SenseVoiceSmall`) |
| `avg_duration_sec` | 벤치마크에 사용된 오디오들의 평균 길이(초). 두 모델 모두 같은 `eval_pairs`로 평가하므로 이 값은 두 모델에서 동일해야 정상 |
| `avg_latency_sec` | 오디오 1개를 텍스트로 변환하는 데 걸린 평균 추론 시간(초). 값이 작을수록 빠름 |
| `avg_rtf` | Real-Time Factor = `avg_latency_sec / avg_duration_sec`. **1보다 작으면 오디오 길이보다 빠르게 처리**한다는 뜻(예: 0.2면 10초 음성을 2초에 처리) — 실시간 스트리밍 적용 가능 여부를 가늠하는 핵심 지표. 1보다 크면 녹음보다 처리가 느려 실시간 사용이 어려움 |
| `avg_cer` | Character Error Rate 평균. 정답 텍스트 대비 삽입/삭제/치환된 음절의 비율(0~1). 0에 가까울수록 정확하고, 예를 들어 0.15면 음절 100개당 약 15개가 틀렸다는 뜻 |

이 프로젝트의 `backend/app/services/stt/`에 실제 모델을 넣을 때는, RTF가 낮으면서 CER도 낮은 쪽이 실시간 스트리밍 채점(`api/websocket.py`의 `partial_transcript` 스트림)에 더 적합하다.